# Aggregate macro surprises by release date

This notebook creates `daily_event_df`, with one row for each official release date. Individual metric surprises are divided by their metric-specific historical standard deviation before averaging, so payroll jobs, percentage points, and basis points are not mixed in raw units.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)

In [2]:
def find_project_root(start: Path) -> Path:
    required = Path("outputs/macro_event_identification/event_metrics_actual_vs_expected.csv")
    for directory in (start.resolve(), *start.resolve().parents):
        if (directory / required).exists():
            return directory
    raise FileNotFoundError(f"Could not find {required}")

ROOT = find_project_root(Path.cwd())
DATA_DIR = ROOT / "outputs" / "macro_event_identification"
METRICS_PATH = DATA_DIR / "event_metrics_actual_vs_expected.csv"
EVENTS_PATH = DATA_DIR / "macro_events.csv"
OUTPUT_PATH = DATA_DIR / "macro_event_surprises_by_release_date.csv"

metrics = pd.read_csv(METRICS_PATH)
events = pd.read_csv(EVENTS_PATH)
metrics[["actual_value", "expected_value", "surprise"]] = metrics[["actual_value", "expected_value", "surprise"]].apply(pd.to_numeric, errors="coerce")
events["official_release_datetime_et"] = pd.to_datetime(events["official_release_datetime_et"], errors="coerce")
print(f"Loaded {len(events)} official events and {metrics['surprise'].notna().sum()} comparable metric surprises.")

Loaded 43 official events and 70 comparable metric surprises.


In [3]:
release_info = events[["event_id", "event_type", "official_release_datetime_et"]].drop_duplicates("event_id")
components = metrics.merge(release_info, on=["event_id", "event_type"], how="left", validate="many_to_one")
components = components.loc[components["surprise"].notna()].copy()
components["release_date"] = components["official_release_datetime_et"].dt.normalize()
components["expectation_phase"] = components["expectation_observation_phase"]

metric_scale = components.groupby("metric")["surprise"].std(ddof=1)
components["metric_surprise_scale"] = components["metric"].map(metric_scale)
components["standardized_surprise"] = np.where(
    components["metric_surprise_scale"].gt(0),
    components["surprise"] / components["metric_surprise_scale"],
    0.0,
)
components["is_pre"] = components["expectation_phase"].eq("pre")
components["surprise_pp"] = components["surprise"].where(components["unit"].eq("percent"))
components["surprise_jobs"] = components["surprise"].where(components["unit"].eq("jobs"))
components["surprise_thousand_jobs"] = components["surprise"].where(components["unit"].eq("thousand_jobs"))
components["surprise_bps"] = components["surprise"].where(components["unit"].eq("basis_points"))

def join_unique(values):
    return " | ".join(sorted(set(values.dropna().astype(str))))

aggregated = components.groupby("release_date", as_index=False).agg(
    event_surprise_mean=("standardized_surprise", "mean"),
    surprise_count=("surprise", "size"),
    pre_surprise_count=("is_pre", "sum"),
    raw_mean_percentage_points=("surprise_pp", "mean"),
    raw_mean_jobs=("surprise_jobs", "mean"),
    raw_mean_thousand_jobs=("surprise_thousand_jobs", "mean"),
    raw_mean_basis_points=("surprise_bps", "mean"),
)
aggregated["post_surprise_count"] = aggregated["surprise_count"] - aggregated["pre_surprise_count"]

pre_aggregated = (
    components.loc[components["is_pre"]]
    .groupby("release_date", as_index=False)
    .agg(event_surprise_pre_mean=("standardized_surprise", "mean"))
)
aggregated = aggregated.merge(pre_aggregated, on="release_date", how="left", validate="one_to_one")

release_scaffold = events.assign(
    release_date=events["official_release_datetime_et"].dt.normalize()
).groupby("release_date", as_index=False).agg(
    official_release_count=("event_id", "nunique"),
    event_ids=("event_id", join_unique),
    event_types=("event_type", join_unique),
)

daily_event_df = (
    release_scaffold.merge(aggregated, on="release_date", how="left", validate="one_to_one")
    .sort_values("release_date")
    .reset_index(drop=True)
)
for column in ["surprise_count", "pre_surprise_count", "post_surprise_count"]:
    daily_event_df[column] = daily_event_df[column].fillna(0).astype(int)

daily_event_df = daily_event_df[[
    "release_date", "event_surprise_mean", "event_surprise_pre_mean",
    "surprise_count", "pre_surprise_count", "post_surprise_count",
    "official_release_count", "event_ids", "event_types",
    "raw_mean_percentage_points", "raw_mean_jobs",
    "raw_mean_thousand_jobs", "raw_mean_basis_points",
]]

assert daily_event_df["release_date"].is_unique
assert daily_event_df["surprise_count"].sum() == metrics["surprise"].notna().sum()

In [4]:
daily_event_df.to_csv(OUTPUT_PATH, index=False)
print(f"daily_event_df shape: {daily_event_df.shape}")
print(f"Dates with a computable event surprise: {daily_event_df['event_surprise_mean'].notna().sum()}")
print(f"Saved to: {OUTPUT_PATH}")
daily_event_df

daily_event_df shape: (43, 13)
Dates with a computable event surprise: 36
Saved to: C:\Users\Jerry Cai\Desktop\NLP_Final_Assignment\outputs\macro_event_identification\macro_event_surprises_by_release_date.csv


,release_date,event_surprise_mean,event_surprise_pre_mean,surprise_count,pre_surprise_count,post_surprise_count,official_release_count,event_ids,event_types,raw_mean_percentage_points,raw_mean_jobs,raw_mean_thousand_jobs,raw_mean_basis_points
0,2025-10-24,-1.188774,NaN,2,0,2,1,CPI_2025-09,CPI_RELEASE,-0.100000,NaN,NaN,NaN
1,2025-10-29,0.000000,0.000000,1,1,0,1,FOMC_2025-10-29,FOMC_RATE_DECISION,NaN,NaN,NaN,0.0
2,2025-11-20,0.911475,NaN,1,0,1,1,EMPLOYMENT_2025-09,EMPLOYMENT_RELEASE,NaN,69000.0,NaN,NaN
3,2025-12-05,NaN,NaN,0,0,0,1,PCE_2025-09,PCE_RELEASE,NaN,NaN,NaN,NaN
4,2025-12-10,0.000000,0.000000,3,3,0,1,FOMC_2025-12-10,FOMC_RATE_DECISION,0.000000,NaN,NaN,0.0
5,2025-12-16,0.669600,NaN,2,0,2,1,EMPLOYMENT_2025-11,EMPLOYMENT_RELEASE,0.100000,19000.0,NaN,NaN
6,2025-12-18,-2.613870,NaN,2,0,2,1,CPI_2025-11,CPI_RELEASE,-0.400000,NaN,NaN,NaN
7,2026-01-09,-0.696020,-0.696020,2,2,0,1,EMPLOYMENT_2025-12,EMPLOYMENT_RELEASE,-0.100000,-23000.0,NaN,NaN
8,2026-01-13,-0.559511,-0.559511,3,3,0,1,CPI_2025-12,CPI_RELEASE,-0.066667,NaN,NaN,NaN
9,2026-01-22,0.000000,0.000000,1,1,0,1,PCE_2025-10_2025-11,PCE_RELEASE,0.000000,NaN,NaN,NaN
